# GHO — Binance Trading Bot

Ejecuta el bot de trading en Google Colab paso a paso.

**IMPORTANTE:**
- Colab gratuito tiene un límite de ~12 horas por sesión.
- Tus API keys **nunca se guardan** en el notebook.
- Puedes usar el testnet de Binance antes de arriesgar dinero real.

---
**Pasos:**
1. Clonar el repositorio
2. Instalar dependencias
3. Ingresar tus API keys
4. Configurar el bot
5. Ejecutar

## Paso 1 — Clonar el repositorio

In [ ]:
import os

REPO_URL = "https://github.com/girr7/GHO.git"
BRANCH   = "claude/binance-trading-bot-p13AI"
REPO_DIR = "/content/GHO"

if os.path.exists(REPO_DIR):
    print("Repositorio ya clonado. Actualizando...")
    os.chdir(REPO_DIR)
    !git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print("\n✅ Repositorio listo en:", os.getcwd())
!ls -la

## Paso 2 — Instalar dependencias

Solo instala paquetes que Colab no trae por defecto (no toca numpy/pandas/requests).

In [ ]:
# Instala solo los paquetes que Colab no incluye por defecto.
# NO se actualizan numpy, pandas ni requests para no romper el entorno de Colab.
!pip install -q --no-deps python-binance==1.0.19
!pip install -q python-dotenv ta schedule colorlog websocket-client
print("\n✅ Dependencias instaladas correctamente.")

## Paso 3 — Ingresar API Keys de Binance

Se pedirán de forma segura (no quedan visibles en el notebook).

> Si usas el **testnet**, obtén tus keys en: https://testnet.binance.vision/

In [ ]:
import getpass

print("Ingresa tus credenciales de Binance:")
api_key    = getpass.getpass("API Key    : ")
api_secret = getpass.getpass("API Secret : ")

if not api_key or not api_secret:
    raise ValueError("Las API keys no pueden estar vacías.")

print("✅ Credenciales recibidas (no se mostrarán ni guardarán).")

## Paso 4 — Configurar el bot

Ajusta los parámetros según tus preferencias.

In [ ]:
# ============================================================
#  CONFIGURACIÓN — edita estos valores a tu gusto
# ============================================================

USE_TESTNET         = "true"       # "true" = testnet | "false" = real
TRADING_PAIR        = "BTCUSDT"    # Par de trading
QUOTE_ASSET         = "USDT"       # Moneda base para operar
TRADE_AMOUNT        = "50"         # Monto por operación (en USDT)
STRATEGY            = "combined"   # "rsi", "ma_crossover" o "combined"
STOP_LOSS_PERCENT   = "2.0"        # % de stop-loss
TAKE_PROFIT_PERCENT = "4.0"        # % de take-profit
MAX_OPEN_TRADES     = "3"          # Máximo de trades abiertos simultáneos
LOG_LEVEL           = "INFO"       # DEBUG, INFO, WARNING, ERROR

# ============================================================
# Crear archivo .env
# ============================================================
import os

env_content = f"""BINANCE_API_KEY={api_key}
BINANCE_API_SECRET={api_secret}
USE_TESTNET={USE_TESTNET}
TRADING_PAIR={TRADING_PAIR}
QUOTE_ASSET={QUOTE_ASSET}
TRADE_AMOUNT={TRADE_AMOUNT}
STRATEGY={STRATEGY}
STOP_LOSS_PERCENT={STOP_LOSS_PERCENT}
TAKE_PROFIT_PERCENT={TAKE_PROFIT_PERCENT}
MAX_OPEN_TRADES={MAX_OPEN_TRADES}
LOG_LEVEL={LOG_LEVEL}
"""

with open("/content/GHO/.env", "w") as f:
    f.write(env_content)

print("✅ Archivo .env creado con la configuración.")
print(f"   Par       : {TRADING_PAIR}")
print(f"   Estrategia: {STRATEGY}")
print(f"   Testnet   : {USE_TESTNET}")
print(f"   Monto     : ${TRADE_AMOUNT} USDT por trade")
print(f"   SL / TP   : {STOP_LOSS_PERCENT}% / {TAKE_PROFIT_PERCENT}%")

## Paso 5 — Ejecutar el bot

El bot corre en un hilo en segundo plano y los logs aparecen en tiempo real.

> Para **detenerlo**: `Runtime → Interrupt execution` (o Ctrl+M I).

In [ ]:
import sys, threading, time, os

os.chdir("/content/GHO")
if "/content/GHO" not in sys.path:
    sys.path.insert(0, "/content/GHO")

from bot import TradingBot

bot_instance = TradingBot()

def run_bot():
    try:
        bot_instance.run()
    except Exception as e:
        print(f"\n❌ El bot se detuvo con error: {e}")

bot_thread = threading.Thread(target=run_bot, daemon=True)
bot_thread.start()
print("✅ Bot iniciado. Logs aparecerán a continuación...")
print("   Para detener: Runtime → Interrupt execution\n")

try:
    while bot_thread.is_alive():
        time.sleep(1)
except KeyboardInterrupt:
    print("\nBot detenido por el usuario.")

## (Opcional) Ver log guardado

In [ ]:
import os
log_path = "/content/GHO/bot.log"
if os.path.exists(log_path):
    with open(log_path) as f:
        print(f.read())
else:
    print("No se encontró archivo de log todavía.")

---

## Links útiles
- Testnet Binance: https://testnet.binance.vision/
- API Management (real): https://www.binance.com/en/my/settings/api-management
- Documentación python-binance: https://python-binance.readthedocs.io/